# LMP-SPARK
**Author:** Ryan J. McLaughlin  
**Date:** 2025-05-06

This is meant to be a complete end-to-end document for running:

1. Amplicon Sequence Variant (ASV) pipeline
2. General statistics
3. Downstream analytics
4. Figure/Table creation

## 1. Amplicon Sequence Variant (ASV) pipeline
This section reviews the steps involved in creating ASVs from raw FASTQ data.

### Setup Environments for running the pipeline

In [ ]:
%%bash
# Define the environment name
ENV_NAME="spark_env"
ENV_YAML="$PWD/spark_env.yaml"

QI_NAME="qiime2-amplicon-2024.10"

# Check if the environment exists
if mamba env list | grep -q "^${ENV_NAME} "; then
    echo "Environment ${ENV_NAME} already exists."
else
    echo "Environment ${ENV_NAME} does not exist. Creating it..."
    mamba env create -y -n ${ENV_NAME} -f ${ENV_YAML}
fi

# Check if the QIIME2 environment exists
if mamba env list | grep -q "^${QI_NAME} "; then
    echo "Environment ${QI_NAME} already exists."
else
    echo "Environment ${QI_NAME} does not exist. Creating it..."
    mamba env create -y -n ${QI_NAME} -c bioconda qiime2-amplicon-2024.10
fi

### Run the ASV pipeline

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

./run_vsearch.sh --skip-fastp --skip-merge --skip-filter --skip-concat --skip-derep --skip-denoise --skip-chimera --skip-swarm --skip-nontarget

### BLAST contamination DB, Mito DB, run MitoMaster

In [ ]:
seqkit split2 -s 100 /home/ryan/Projects/UBC/LMP/SPARK_data/kits_vsearch_output/ASVs/ASVs.fasta
mkdir -p /home/ryan/Projects/UBC/LMP/SPARK_data/kits_vsearch_output/mitomap
python mitomaster.py
blastn -query /home/ryan/Projects/UBC/LMP/SPARK_data/kits_vsearch_output/ASVs/ASVs.fasta -db /home/ryan/Projects/UBC/LMP/SPARK_data/ref_db/mito_ncbi -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" -out /home/ryan/Projects/UBC/LMP/SPARK_data/kits_vsearch_output/mitomap/mito_ncbi.blast6.tsv
blastn -query /home/ryan/Projects/UBC/LMP/SPARK_data/kits_vsearch_output/ASVs/ASVs.fasta -db /home/ryan/Projects/UBC/LMP/SPARK_data/ref_db/ssu_pipeline_contaminants -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" -out /home/ryan/Projects/UBC/LMP/SPARK_data/kits_vsearch_output/mitomap/ssu_pipeline_contaminants.blast6.tsv


### Run General Statistics

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env
THREADS="$(nproc --all)"

seqkit stat -a -T -o ../SPARK_data/kits_vsearch_output/stats/fastq_stats.tsv -j ${THREADS} ../SPARK_data/fastq_kits/*
seqkit stat -a -T -o ../SPARK_data/kits_vsearch_output/stats/fastp_fastqs.tsv -j ${THREADS} ../SPARK_data/kits_vsearch_output/fastp/*.fastq.gz
seqkit stat -a -T -o ../SPARK_data/kits_vsearch_output/stats/merged_fastqs.tsv -j ${THREADS} ../SPARK_data/kits_vsearch_output/merged/*.fastq
seqkit stat -a -T -o ../SPARK_data/kits_vsearch_output/stats/filtered_fastqs.tsv -j ${THREADS} ../SPARK_data/kits_vsearch_output/filtered/*.fasta
seqkit stat -a -T -o ../SPARK_data/kits_vsearch_output/stats/concat_fastas.tsv -j ${THREADS} ../SPARK_data/kits_vsearch_output/concat/concat.fasta

### Run QIIME2 Taxonomic Classifier

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate qiime2-amplicon-2024.10

python qiime_vs_classifier.py \
  --input-fasta SPARK_data/vsearch_output/ASVs/ASV_filtered.micro.fasta \
  --ref-taxonomy SPARK_data/ref_db/silva-138_2-ssu-nr99-tax.qza \
  --ref-seqs SPARK_data/ref_db/silva-138_2-ssu-nr99-seqs-DNA.qza \
  --output-tsv vsearch_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
  --stats-output vsearch_output/taxonomy/ASV_SILVA_stats.full-length.vsearch.tsv

### Build Sankey Diagram

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python sankey_builder.py

### Plot Metadata

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_metadata.py

### Plot Upset

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_upset.py
python venn_bubbles.py

### Run Alpha and Beta Diversity

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python calc_div.py
python plot_diversity.py

### Run indicspecies (R)

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env
Rscript run_indicspecies.R

### Plot indicspecies Results

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_indicspecies.py

### Plot Clustermaps

In [20]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_clustermaps.py

### Run SPIEC-EASI (R)

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

Rscript run_spieceasi.R

### Graph Network

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python graph_network.py

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

Rscript run_spieceasi_multi.R